# QuantumStega — Q-series + DPM-Solver++ (Q1–Q6)

**Bản nâng cấp tự chứa** của `quantumstega-bb84-key` (file gốc giữ nguyên). Tích hợp 6 cải tiến
"chưa có tiền lệ" vào MỘT pipeline CRoSS + Quantum:

| Mã | Tên | Ý nghĩa |
|----|-----|---------|
| **DPM++** | CV-Symplectic + **DPM-Solver++ sampler** | DDIM bậc 1 → DPM-Solver++ bậc 2: ít sai số inversion, **25 bước thay 50**, PSNR cao hơn. |
| **Q1** | CV-Symplectic Scrambler | tổng quát hoá orthogonal-scramble lên nhóm symplectic + squeezing (Bloch-Messiah); khả nghịch chính xác. |
| **Q2** | Helstrom-bounded undetectability | passive ⇒ phân phối latent TRÙNG ⇒ warden ở mức 1/2 (**bảo mật hoàn hảo, có chứng minh**). |
| **Q3** | Decoherence-Free Subspace probe | dò không-gian-con latent bền JPEG để đặt payload thụ động (robustness có cấu trúc). |
| **Q4** | Quantum-Diffusion generator | tham số biến đổi latent **sinh từ VQC PennyLane** (quantum thật, không trơ). |
| **Q5** | Quantum-money token | token container **bất khả giả** (no-cloning) — vượt HMAC. |
| **Q6** | SWAP-test fingerprint | so khớp container n-bit với **O(log n)** chiều (lợi thế truyền thông). |

**Cách chạy:** Run All. Cell *Self-test* chạy ngay trên CPU (chứng minh Q1–Q6, 35/35). Cell *Run pipeline*
cần GPU + Stable Diffusion v1.5. Đổi `image_path` trong `QSERIES_CFG` sang ảnh secret của bạn.

> Lưu ý trung thực: trên phần cứng cổ điển đây là **mô phỏng** các nguyên lý lượng tử
> (prepare-measure, no-cloning, CHSH). Giá trị nằm ở **mô hình bảo mật & cấu trúc toán học**, không phải tốc độ.


In [ ]:
# ============ MÔI TRƯỜNG: Python 3.10 + pip (online) ============
!apt-get update -y
!apt-get install python3.10 python3.10-distutils -y
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10


In [ ]:
!python3.10 --version


In [ ]:
!python3.10 -m pip install \
    numpy \
    scipy \
    pandas \
    scikit-learn==1.7.1 \
    imbalanced-learn==0.14.1 \
    torch==2.9.1 \
    tensorflow==2.14.0 \
    keras==2.14.0 \
    autograd==1.8.0 \
    pykan==0.2.8 \
    pennylane==0.42.3 \
    pennylane-lightning==0.42.0 \
    autoray>=0.7.0 \
    matplotlib==3.10.8 \
    seaborn==0.13.2 \
    plotly==6.5.2 \
    tqdm==4.67.2 \
    joblib==1.5.3 \
    opencv-python \
    einops \
    matplotlib-inline \
    ipython \
    PyYAML

# >>> BỔ SUNG cho Stable Diffusion / CRoSS — pipeline Q-series BẮT BUỘC cần <<<
!python3.10 -m pip install diffusers transformers accelerate safetensors pillow


In [ ]:
!python3.10 -m pip install matplotlib-inline ipython


In [ ]:
%%writefile imports.py
# ============================================================================
# MODULE: CENTRALIZED IMPORTS & GLOBAL SEED CONFIGURATION (WITH BACKEND FIX)
# ============================================================================

# Ep Matplotlib dung 'Agg' truoc khi bat ky module nao import no
import os
import matplotlib
matplotlib.use('Agg')

# --- Core Systems ---
import sys
import glob
import copy
import random
from pathlib import Path

# --- Numerical & Scientific Computing ---
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist, directed_hausdorff
from scipy.ndimage import binary_erosion

# --- Computer Vision & Visualization ---
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

# --- PyTorch Ecosystem ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, Subset

# --- Advanced Architectures & Quantum Elements ---
from kan import KAN
import pennylane as qml
from einops import rearrange

# --- Scikit-Learn: Preprocessing & Feature Engineering ---
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, label_binarize
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# --- Scikit-Learn: Validation & Sampling ---
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, GroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.utils import resample

# --- Scikit-Learn: Metric Evaluations ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, balanced_accuracy_score,
    average_precision_score, matthews_corrcoef,
    cohen_kappa_score, brier_score_loss,
    roc_auc_score, roc_curve, auc, classification_report
)

# --- Utilities ---
from tqdm import tqdm
import importlib

# ============================================================
# GLOBAL REPRODUCIBILITY SEED CONFIGURATION
# ============================================================
random_state = 42
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [11]:
import os
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, Union

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageOps
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from diffusers import StableDiffusionPipeline, DDIMScheduler
from tqdm import tqdm

# Kaggle paths
DEFAULT_IMAGE_PATH = "/kaggle/input/asserts/1.png"
DEFAULT_OUTPUT_DIR = "/kaggle/working/output"
DEFAULT_TRAIN_DATA_DIR = "/kaggle/input/asserts"
DEFAULT_CKPT_DIR = "/kaggle/working/quantum_ckpts"

# Stable Diffusion defaults
MY_TOKEN = ""
LOW_RESOURCE = False
GUIDANCE_SCALE = 1.0
MAX_NUM_WORDS = 77
MODEL_ID = "runwayml/stable-diffusion-v1-5"

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)
print("Default input image:", DEFAULT_IMAGE_PATH)
print("Default output dir:", DEFAULT_OUTPUT_DIR)

`CLIPImageProcessor` requires torchvision (not installed); falling back to `CLIPImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `CLIPImageProcessorPil` directly to silence this warning.
`SiglipImageProcessor` requires torchvision (not installed); falling back to `SiglipImageProcessorPil` for backward compatibility. Install torchvision to use the default backend, or import `SiglipImageProcessorPil` directly to silence this warning.


Device: cuda:0
Default input image: /kaggle/input/asserts/1.png
Default output dir: /kaggle/working/output


In [ ]:
# =========================
# Stable Diffusion setup (CRoSS DDIM scheduler) — dùng cho pipeline Q-series
# =========================
from diffusers import (StableDiffusionPipeline, DDIMScheduler,
                       DPMSolverMultistepScheduler,
                       DPMSolverMultistepInverseScheduler)

scheduler = DDIMScheduler(
    beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear",
    clip_sample=False, set_alpha_to_one=False,
)

def build_pipeline(model_id=MODEL_ID, token=MY_TOKEN):
    try:
        if token:
            pipe = StableDiffusionPipeline.from_pretrained(model_id, token=token, scheduler=scheduler).to(device)
        else:
            pipe = StableDiffusionPipeline.from_pretrained(model_id, scheduler=scheduler).to(device)
    except TypeError:
        pipe = StableDiffusionPipeline.from_pretrained(model_id, scheduler=scheduler).to(device)
    try:
        pipe.disable_xformers_memory_efficient_attention()
    except AttributeError:
        pass
    return pipe

print("SD setup OK. GUIDANCE_SCALE =", GUIDANCE_SCALE, "| device =", device)
print("DPM-Solver++ available:", hasattr(__import__("diffusers"), "DPMSolverMultistepInverseScheduler"))


## Engine Q-series (tự chứa — numpy/scipy/hashlib + PennyLane optional)


In [ ]:
"""
qseries_core.py
===============
Engine lõi (framework-independent: numpy/scipy/hashlib) cho **QuStega Q-series** — thế hệ
cải tiến 2 của pipeline CRoSS-Quantum, vượt lên trên C1-C8 bằng cách biến "lượng tử" thành
ĐỊNH LÝ / CẤU TRÚC chứ không chỉ là module ghép thêm.

Insight nền: latent noise xT ~ N(0, sigma^2 I) CHÍNH LÀ một trạng thái Gaussian đa-mode
(continuous-variable quantum state). Do đó:
  - Q1  CV-Symplectic Scrambler : tổng quát hoá orthogonal-scramble (C1) lên TOÀN nhóm
        symplectic Sp(2n,R) bằng phân rã Bloch-Messiah (passive O1 -> squeeze -> passive O2).
        Passive = bảo toàn N(0,sigma^2 I) chính xác; squeezing = núm đánh đổi vô-hình/keyspace.
  - Q2  Helstrom-Bounded Undetectability : đặt bài toán của warden (steganalyzer) thành
        phân biệt 2 trạng thái Gaussian; passive => phân phối TRÙNG => warden ở mức 1/2
        (bảo mật hoàn hảo Cachin, CÓ CHỨNG MINH). Squeeze => chặn trên qua Hellinger/Pinsker.
  - Q3  Decoherence-Free Subspace embedding : mô hình hoá kênh tấn công (JPEG...) như kênh
        nhiễu; nhúng payload vào không-gian-con bền nhất (thụ động) thay vì chỉ tăng ECC (C6).
  - Q4  Quantum-Diffusion Scrambler : chuỗi unitary có khoá (QuDDPM-style "thay nhiễu Gaussian
        bằng unitary") -> "quantum diffusion" tác động THẬT lên latent (fallback numpy ở đây;
        VQC PennyLane ở qseries_sd.py).
  - Q5  Quantum-Money token : token bất khả giả (no-cloning) cho container; vượt HMAC (C4).
  - Q6  SWAP-test Fingerprint : so khớp container n-bit với O(log n) "qubit" -> lợi thế
        ĐỘ PHỨC TẠP TRUYỀN THÔNG (provable), trả lời "advantage ở đâu khi chạy simulator".

Mọi randomness có khoá đều đi qua SHAKE-256 (HKDF-style). Trên phần cứng cổ điển đây là
MÔ PHỎNG các nguyên lý lượng tử; giá trị nằm ở MÔ HÌNH BẢO MẬT, không phải tốc độ.

Chạy self-test:  python qseries_core.py
"""
from __future__ import annotations

import base64
import hashlib
import hmac
import json
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np

SQRT2 = float(np.sqrt(2.0))

# =====================================================================================
# 0. KEY DERIVATION (đa khoá Kerckhoffs, B3) + QRNG (C5)
# =====================================================================================

def shake(data: bytes, info: str, nbytes: int) -> bytes:
    """SHAKE-256 dùng như XOF (extendable output) cho HKDF-style derivation."""
    return hashlib.shake_256(data + b"|" + info.encode("utf-8")).digest(nbytes)


def derive_subkey(master: bytes, info: str, nbytes: int = 32) -> bytes:
    return shake(master, "subkey:" + info, nbytes)


def rng_from(key: bytes, info: str) -> np.random.Generator:
    """numpy Generator xác định từ (key, info)."""
    s = int.from_bytes(shake(key, "rng:" + info, 32), "big")
    return np.random.default_rng(s)


def keystream_bits(key: bytes, info: str, nbits: int) -> np.ndarray:
    """Keystream nhị phân độ dài nbits từ SHAKE (OTP/stream cipher)."""
    nbytes = (nbits + 7) // 8
    raw = shake(key, "keystream:" + info, nbytes)
    return np.unpackbits(np.frombuffer(raw, dtype=np.uint8))[:nbits].astype(np.uint8)


def qrng_bits(nbits: int, sim_seed: Optional[int] = None) -> np.ndarray:
    """C5 — Quantum RNG: phần cứng thật = đo |+> trong basis Z (Bernoulli 0.5). Ở đây mô phỏng."""
    return np.random.default_rng(sim_seed).integers(0, 2, size=nbits, dtype=np.uint8)


# =====================================================================================
# 1. QKD CHANNEL MONITOR — BB84 (C2) & E91+CHSH (C4)
# =====================================================================================

@dataclass
class ChannelReport:
    protocol: str
    qber: float = 0.0
    chsh_S: float = 0.0
    intercepted: bool = False
    n_sift: int = 0
    secure: bool = True
    note: str = ""


def bb84_simulate(passphrase: str, salt: bytes, n_qubits: int = 2048,
                  eve_rate: float = 0.0, use_qrng: bool = True) -> Tuple[bytes, ChannelReport]:
    """Mô phỏng BB84 prepare-and-measure làm CHANNEL MONITOR. Eve intercept-resend -> QBER tăng."""
    master = hashlib.sha3_256(b"bb84|" + salt + b"|" + passphrase.encode()).digest()
    rng = rng_from(master, "bb84")
    a_bits = (qrng_bits(n_qubits, sim_seed=int.from_bytes(master[:4], "big"))
              if use_qrng else rng.integers(0, 2, n_qubits, dtype=np.uint8))
    a_bases = rng.integers(0, 2, n_qubits, dtype=np.uint8)
    b_bases = rng.integers(0, 2, n_qubits, dtype=np.uint8)

    b_bits = a_bits.copy()
    diff_basis = a_bases != b_bases
    b_bits[diff_basis] = rng.integers(0, 2, int(diff_basis.sum()), dtype=np.uint8)

    eve_intercepted = False
    if eve_rate > 0.0:
        eve_rng = np.random.default_rng(0xE7E)
        intercept = eve_rng.random(n_qubits) < eve_rate
        eve_bases = eve_rng.integers(0, 2, n_qubits, dtype=np.uint8)
        for idx in np.nonzero(intercept)[0]:
            eve_val = a_bits[idx] if eve_bases[idx] == a_bases[idx] else int(eve_rng.integers(0, 2))
            b_bits[idx] = eve_val if b_bases[idx] == eve_bases[idx] else int(eve_rng.integers(0, 2))
        eve_intercepted = bool(intercept.any())

    sift = a_bases == b_bases
    a_sift, b_sift = a_bits[sift], b_bits[sift]
    n_sift = int(sift.sum())
    half = n_sift // 2
    qber = float(np.mean(a_sift[:half] != b_sift[:half])) if half > 0 else 0.0

    raw_key_bits = a_sift[half:]
    raw_bytes = np.packbits(raw_key_bits).tobytes() if raw_key_bits.size else b"\x00"
    sifted_key = hashlib.sha3_256(b"pa|" + raw_bytes + b"|" + master).digest()
    return sifted_key, ChannelReport("bb84", qber=qber, intercepted=eve_intercepted, n_sift=n_sift)


def _chsh_corr(alpha: float, beta: float, n: int, rng: np.random.Generator,
               eve_rate: float, eve_rng: Optional[np.random.Generator]) -> float:
    E = np.cos(2.0 * (alpha - beta))
    A = rng.integers(0, 2, n) * 2 - 1
    same = rng.random(n) < (1.0 + E) / 2.0
    B = np.where(same, A, -A)
    if eve_rate > 0.0 and eve_rng is not None:
        intercept = eve_rng.random(n) < eve_rate
        eve_angles = np.array([0.0, np.pi / 4])
        e = eve_angles[eve_rng.integers(0, 2, n)]
        rE = rng.integers(0, 2, n) * 2 - 1
        E1, E2 = np.cos(2.0 * (alpha - e)), np.cos(2.0 * (e - beta))
        A_e = np.where(rng.random(n) < (1 + E1) / 2, rE, -rE)
        B_e = np.where(rng.random(n) < (1 + E2) / 2, rE, -rE)
        A = np.where(intercept, A_e, A)
        B = np.where(intercept, B_e, B)
    return float(np.mean(A * B))


def e91_simulate(passphrase: str, salt: bytes, n_pairs: int = 4096,
                 eve_rate: float = 0.0) -> Tuple[bytes, ChannelReport]:
    """E91: dùng CHSH phát hiện nghe lén. Không Eve: |S|~2.83. Có Eve: |S|<=2."""
    master = hashlib.sha3_256(b"e91|" + salt + b"|" + passphrase.encode()).digest()
    rng = rng_from(master, "e91")
    eve_rng = np.random.default_rng(0xE91E) if eve_rate > 0 else None
    a1, a2 = 0.0, np.pi / 4
    b1, b2 = np.pi / 8, 3 * np.pi / 8
    per = max(256, n_pairs // 4)
    S = (_chsh_corr(a1, b1, per, rng, eve_rate, eve_rng)
         - _chsh_corr(a1, b2, per, rng, eve_rate, eve_rng)
         + _chsh_corr(a2, b1, per, rng, eve_rate, eve_rng)
         + _chsh_corr(a2, b2, per, rng, eve_rate, eve_rng))
    sifted_key = hashlib.sha3_256(b"e91pa|" + master).digest()
    return sifted_key, ChannelReport("e91", chsh_S=float(S), intercepted=eve_rate > 0)


# =====================================================================================
# 2. QUANTUM KEY MANAGER — gom QKD + đa khoá + gate tamper (C2/C4/C5/B3/C6)
# =====================================================================================

@dataclass
class QuantumKeyManager:
    passphrase: str
    protocol: str = "bb84"
    use_qrng: bool = True
    salt: bytes = field(default_factory=lambda: b"\x00" * 16)
    eve_rate: float = 0.0
    qber_threshold: float = 0.11
    chsh_threshold: float = 2.4

    sifted_key: bytes = b""
    report: Optional[ChannelReport] = None

    def establish(self) -> ChannelReport:
        if self.protocol == "e91":
            self.sifted_key, rep = e91_simulate(self.passphrase, self.salt, eve_rate=self.eve_rate)
            rep.secure = abs(rep.chsh_S) >= self.chsh_threshold
            if not rep.secure:
                rep.note = f"CHSH |S|={abs(rep.chsh_S):.3f} < {self.chsh_threshold} => nghi nghe len."
        else:
            self.sifted_key, rep = bb84_simulate(self.passphrase, self.salt,
                                                  eve_rate=self.eve_rate, use_qrng=self.use_qrng)
            rep.secure = rep.qber <= self.qber_threshold
            if not rep.secure:
                rep.note = f"QBER={rep.qber:.3f} > {self.qber_threshold} => nghi nghe len."
        self.report = rep
        return rep

    # --- đa khoá (B3) ---
    def subkey(self, name: str, nbytes: int = 32) -> bytes:
        assert self.sifted_key, "Gọi establish() trước."
        return derive_subkey(self.sifted_key, name, nbytes)

    def K_seed(self) -> bytes:  return self.subkey("scrambler-seed")
    def K_qw(self) -> bytes:    return self.subkey("quantum-walk")
    def K_enc(self) -> bytes:   return self.subkey("otp-enc")
    def K_conf(self) -> bytes:  return self.subkey("container-auth")
    def K_money(self) -> bytes: return self.subkey("quantum-money")   # Q5
    def K_dfs(self) -> bytes:   return self.subkey("dfs-layout")      # Q3

    def adapt(self) -> Dict[str, float]:
        """C6 — QBER-adaptive: kênh càng nhiễu => càng nhiều ECC / steps / ngưỡng chặt."""
        q = self.report.qber if (self.report and self.protocol == "bb84") else 0.0
        if q < 0.02:
            return {"num_steps": 50, "ecc_repeat": 1, "check_rate": 0.12}
        elif q < 0.06:
            return {"num_steps": 50, "ecc_repeat": 3, "check_rate": 0.15}
        return {"num_steps": 75, "ecc_repeat": 5, "check_rate": 0.20}


# =====================================================================================
# 3. QUANTUM-WALK S-BOX -> latent permutation (C7)
# =====================================================================================

def quantum_walk_distribution(n_nodes: int, steps: int, coin_bias: float = 0.5,
                              phase: float = 0.0) -> np.ndarray:
    psi = np.zeros((n_nodes, 2), dtype=np.complex128)
    psi[0, 0] = 1.0 / np.sqrt(2); psi[0, 1] = 1j / np.sqrt(2)
    c, s, ph = np.sqrt(coin_bias), np.sqrt(1 - coin_bias), np.exp(1j * phase)
    C = np.array([[c, s * ph], [s * np.conj(ph), -c]], dtype=np.complex128)
    for _ in range(steps):
        psi = psi @ C.T
        psi = np.stack([np.roll(psi[:, 0], -1), np.roll(psi[:, 1], +1)], axis=1)
    prob = np.abs(psi[:, 0]) ** 2 + np.abs(psi[:, 1]) ** 2
    return prob / prob.sum()


def quantum_walk_permutation(key: bytes, length: int) -> np.ndarray:
    """Hoán vị có khoá từ quantum walk (phép orthogonal => bảo toàn Gaussian, khả nghịch)."""
    rng = rng_from(key, "qw-params")
    steps = int(rng.integers(20, 60))
    coin_bias = float(rng.uniform(0.3, 0.7))
    phase = float(rng.uniform(0, 2 * np.pi))
    n_nodes = max(8, int(length))
    prob = quantum_walk_distribution(n_nodes, steps, coin_bias, phase)
    jitter = rng_from(key, "qw-jitter").random(n_nodes) * 1e-9
    return np.argsort(np.argsort((prob + jitter)[:length])).astype(np.int64)


def apply_permutation(x_flat: np.ndarray, perm: np.ndarray) -> np.ndarray:
    return x_flat[perm]


def invert_permutation(perm: np.ndarray) -> np.ndarray:
    inv = np.empty_like(perm)
    inv[perm] = np.arange(perm.size)
    return inv


# =====================================================================================
# 4a. ORTHOGONAL LATENT SCRAMBLER (C1 — passive Gaussian unitary)
# =====================================================================================

def haar_orthogonal(d: int, key: bytes, info: str = "ortho") -> np.ndarray:
    """Haar-orthogonal d×d từ khoá (QR của Gaussian, chuẩn hoá dấu — Mezzadri 2007)."""
    g = rng_from(key, info).standard_normal((d, d))
    Q, R = np.linalg.qr(g)
    return (Q * np.sign(np.diag(R))).astype(np.float64)


class OrthogonalScrambler:
    """C1 — biến đổi orthogonal Q trên latent theo khối. U^{-1}=U^T (khả nghịch exact);
    bảo toàn N(0,sigma^2 I) (đẳng hướng); cond=1 (không khuếch đại sai số inversion)."""

    def __init__(self, key: bytes, block: int = 64):
        self.key = key
        self.block = int(block)
        self.Q = haar_orthogonal(self.block, key, "ortho")

    def _reshape(self, x_flat: np.ndarray) -> Tuple[np.ndarray, int]:
        n = x_flat.size
        pad = (-n) % self.block
        if pad:
            x_flat = np.concatenate([x_flat, np.zeros(pad, dtype=x_flat.dtype)])
        return x_flat.reshape(-1, self.block), n

    def forward(self, x_flat: np.ndarray) -> np.ndarray:
        blocks, n = self._reshape(np.asarray(x_flat, dtype=np.float64))
        return (blocks @ self.Q.T).reshape(-1)[:n]

    def inverse(self, y_flat: np.ndarray) -> np.ndarray:
        blocks, n = self._reshape(np.asarray(y_flat, dtype=np.float64))
        return (blocks @ self.Q).reshape(-1)[:n]


# =====================================================================================
# 4b. ⭐ Q1 — CV-SYMPLECTIC SCRAMBLER (Bloch-Messiah: passive O1 -> squeeze -> passive O2)
# =====================================================================================

class SymplecticScrambler:
    """
    Q1 — tổng quát hoá C1 lên nhóm symplectic Sp(2n,R) qua phân rã Bloch-Messiah:
        S = O2 @ diag(sq) @ O1,   sq = (e^{r_i}, e^{-r_i}) cho mode i (cặp chiều 2i,2i+1).
    - O1, O2: Haar-orthogonal (passive Gaussian unitary) từ khoá.
    - squeeze r_i: từ khoá, scale bởi `squeeze_scale`. squeeze_scale=0 => S orthogonal (≡ C1).
    Khả nghịch CHÍNH XÁC:  S^{-1} = O1^T @ diag(1/sq) @ O2^T.
    Vai trò CV-quantum: O = Gaussian unitary thụ động (bảo toàn vacuum N(0,σ²I));
    squeeze = phép active (đổi covariance) => núm đánh đổi vô-hình (Q2) / keyspace.
    Áp dụng theo KHỐI trên latent đã làm phẳng (block phải CHẴN để ghép mode).
    """

    def __init__(self, key: bytes, block: int = 64, squeeze_scale: float = 0.0):
        if block % 2 != 0:
            raise ValueError("block phải chẵn để ghép mode (q,p).")
        self.key = key
        self.block = int(block)
        self.squeeze_scale = float(squeeze_scale)
        self.O1 = haar_orthogonal(self.block, key, "sympl-O1")
        self.O2 = haar_orthogonal(self.block, key, "sympl-O2")
        n_modes = self.block // 2
        r = rng_from(key, "sympl-squeeze").uniform(-1.0, 1.0, size=n_modes) * self.squeeze_scale
        sq = np.empty(self.block, dtype=np.float64)
        sq[0::2] = np.exp(r)
        sq[1::2] = np.exp(-r)
        self.sq = sq
        self.inv_sq = 1.0 / sq
        self.r = r

    def _reshape(self, x_flat: np.ndarray) -> Tuple[np.ndarray, int]:
        n = x_flat.size
        pad = (-n) % self.block
        if pad:
            x_flat = np.concatenate([x_flat, np.zeros(pad, dtype=x_flat.dtype)])
        return x_flat.reshape(-1, self.block), n

    def forward(self, x_flat: np.ndarray) -> np.ndarray:
        blocks, n = self._reshape(np.asarray(x_flat, dtype=np.float64))
        out = (blocks @ self.O1.T) * self.sq        # O1 @ x, rồi squeeze per-mode
        out = out @ self.O2.T                        # O2 @ (.)
        return out.reshape(-1)[:n]

    def inverse(self, y_flat: np.ndarray) -> np.ndarray:
        blocks, n = self._reshape(np.asarray(y_flat, dtype=np.float64))
        out = (blocks @ self.O2) * self.inv_sq       # O2^T @ y, chia squeeze
        out = out @ self.O1                          # O1^T @ (.)
        return out.reshape(-1)[:n]

    def block_covariance(self, sigma2: float = 1.0) -> np.ndarray:
        """Covariance của block sau biến đổi nếu input ~ N(0, sigma2 I): Σ = sigma2 · S S^T."""
        S = self.O2 @ np.diag(self.sq) @ self.O1
        return sigma2 * (S @ S.T)


# =====================================================================================
# 4c. ⭐ Q1-SPATIAL — CHANNEL-MIXING SCRAMBLER (giữ cấu trúc không gian -> container TỰ NHIÊN)
# =====================================================================================

class ChannelMixingScrambler:
    """
    Q1 (biến thể spatial-coherent) — trộn KÊNH latent bằng orthogonal C×C, ÁP ĐỒNG NHẤT cho
    MỌI vị trí (h,w). Vì phép quay KHÔNG phụ thuộc vị trí, **cấu trúc/tương quan KHÔNG GIAN của
    latent được bảo toàn** => U-Net sinh container TỰ NHIÊN (khác hẳn permutation toàn cục /
    block-scramble vốn phá bố cục => ảnh kỳ lạ).

    Vẫn giữ trọn Q1/Q2:
      - orthogonal (Qc^T Qc = I) => khả nghịch CHÍNH XÁC.
      - bảo toàn N(0,σ²I): mỗi vị trí, vector kênh ~ N(0,σ²I_C) bị quay => vẫn N(0,σ²I_C)
        => Helstrom passive (Q2 đúng, đo trên KHÔNG GIAN KÊNH).
    Hoạt động trên latent shape (...,C,H,W) (numpy). Khoá = Qc dẫn xuất từ key (Q4: từ VQC).
    """

    def __init__(self, key: bytes, channels: int = 4):
        self.channels = int(channels)
        self.block = int(channels)   # "block" = không gian kênh (để Helstrom dùng chung API)
        self.Qc = haar_orthogonal(self.channels, key, "channel-mix")

    def forward(self, x: np.ndarray) -> np.ndarray:
        return np.einsum("ij,...jhw->...ihw", self.Qc, np.asarray(x, np.float64))

    def inverse(self, y: np.ndarray) -> np.ndarray:
        return np.einsum("ij,...jhw->...ihw", self.Qc.T, np.asarray(y, np.float64))

    def block_covariance(self, sigma2: float = 1.0) -> np.ndarray:
        """Covariance trên KHÔNG GIAN KÊNH (C×C): sigma2·Qc Qc^T = sigma2·I => passive."""
        return sigma2 * (self.Qc @ self.Qc.T)


# =====================================================================================
# 5. CONJUGATE-CODING DENIABILITY (C3)
# =====================================================================================

class DeniableMultiplexer:
    """C3 — một container, nhiều ảnh theo khoá. container = decoy + beta·P_real(real-decoy)."""

    def __init__(self, key_real: bytes, dim: int, subspace_ratio: float = 0.5, beta: float = 0.6):
        self.beta = float(beta)
        k = max(1, int(dim * subspace_ratio))
        idx = rng_from(key_real, "deniable-subspace").permutation(dim)[:k]
        self.mask = np.zeros(dim, dtype=bool)
        self.mask[idx] = True

    def mux(self, real_flat: np.ndarray, decoy_flat: np.ndarray) -> np.ndarray:
        real_flat = np.asarray(real_flat, np.float64); decoy_flat = np.asarray(decoy_flat, np.float64)
        out = decoy_flat.copy()
        out[self.mask] = decoy_flat[self.mask] + self.beta * real_flat[self.mask]
        return out

    def demux_real(self, container_flat: np.ndarray, decoy_flat: np.ndarray) -> np.ndarray:
        container_flat = np.asarray(container_flat, np.float64); decoy_flat = np.asarray(decoy_flat, np.float64)
        real = np.zeros_like(container_flat)
        real[self.mask] = (container_flat[self.mask] - decoy_flat[self.mask]) / self.beta
        return real


# =====================================================================================
# 6. ⭐⭐ Q2 — HELSTROM-BOUNDED UNDETECTABILITY (chứng minh tính bất khả phát hiện)
# =====================================================================================
#
# Đặt bài toán warden (steganalyzer): phân biệt H0 "latent tự nhiên ~ N(0,Σ0)" với
# H1 "latent đã giấu tin ~ N(0,Σ1)". Xác suất phát hiện tối ưu (prior 1/2) bị chặn bởi
# tổng biến phân:  P_detect* = 1/2 (1 + TV(P0,P1))   [classical analog của biên Helstrom,
# vì ||rho0-rho1||_1 = 2·TV với trạng thái cổ điển].
#   - passive (Q1, squeeze=0): Σ1 = Σ0 => TV=0 => P_detect*=1/2 (CHỨNG MINH bảo mật hoàn hảo).
#   - squeeze>0: dùng Hellinger (đóng dạng cho Gaussian) chặn TV: H^2 <= TV <= H·sqrt(2-H^2).

def gaussian_kl(S0: np.ndarray, S1: np.ndarray) -> float:
    """KL(N(0,S0) || N(0,S1)) cho 2 covariance SPD (zero-mean)."""
    d = S0.shape[0]
    S1inv = np.linalg.inv(S1)
    _, ld0 = np.linalg.slogdet(S0)
    _, ld1 = np.linalg.slogdet(S1)
    return float(0.5 * (np.trace(S1inv @ S0) - d + (ld1 - ld0)))


def gaussian_hellinger(S0: np.ndarray, S1: np.ndarray) -> float:
    """Khoảng cách Hellinger H in [0,1] giữa N(0,S0) và N(0,S1) (đóng dạng, đối xứng)."""
    _, ld0 = np.linalg.slogdet(S0)
    _, ld1 = np.linalg.slogdet(S1)
    _, ldm = np.linalg.slogdet((S0 + S1) / 2.0)
    log_bc = 0.25 * ld0 + 0.25 * ld1 - 0.5 * ldm   # log của hệ số Bhattacharyya
    bc = float(np.exp(log_bc))
    h2 = max(0.0, 1.0 - bc)
    return float(np.sqrt(h2))


@dataclass
class HelstromReport:
    tv_lower: float          # chặn dưới total variation
    tv_upper: float          # chặn trên total variation
    kl: float
    hellinger: float
    p_detect_upper: float    # chặn trên xác suất phát hiện của warden tối ưu
    advantage_upper: float   # = p_detect_upper - 0.5 (lợi thế tối đa của warden)
    is_passive: bool
    note: str = ""


def helstrom_undetectability(S0: np.ndarray, S1: np.ndarray, atol: float = 1e-9) -> HelstromReport:
    """Báo cáo biên bất-khả-phát-hiện giữa latent tự nhiên (Σ0) và latent đã giấu (Σ1)."""
    if np.allclose(S0, S1, atol=atol):
        return HelstromReport(0.0, 0.0, 0.0, 0.0, 0.5, 0.0, True,
                              "PASSIVE: Σ1=Σ0 => TV=0 => warden ở mức 1/2 (bảo mật hoàn hảo Cachin).")
    kl = gaussian_kl(S0, S1)
    H = gaussian_hellinger(S0, S1)
    tv_lower = H * H
    tv_upper = min(1.0, min(H * np.sqrt(max(0.0, 2.0 - H * H)), np.sqrt(max(0.0, kl) / 2.0)))
    return HelstromReport(tv_lower, tv_upper, kl, H, 0.5 * (1 + tv_upper),
                          0.5 * tv_upper, False,
                          "ACTIVE (squeeze>0): warden bị chặn trên qua Hellinger/Pinsker.")


def empirical_detect_auc(S_scrambler, sigma: float = 1.0, n: int = 4000,
                         seed: int = 0) -> float:
    """
    Thực nghiệm: sinh n block latent THẬT ~ N(0,σ²I) và n block ĐÃ SCRAMBLE; chấm điểm bằng
    LRT tối ưu (dạng toàn phương) rồi tính ĐỘ CHÍNH XÁC TỐT NHẤT của warden = max(AUC, 1-AUC)
    (warden tự chọn chiều quyết định tốt hơn). Passive => ≈0.5 (warden mù); active => >0.5.
    """
    d = S_scrambler.block
    rng = np.random.default_rng(seed)
    real = rng.standard_normal((n, d)) * sigma
    base = rng.standard_normal((n, d)) * sigma
    scr = np.stack([S_scrambler.forward(base[i]) for i in range(n)])  # mỗi hàng là 1 block
    S0 = sigma ** 2 * np.eye(d)
    S1 = S_scrambler.block_covariance(sigma ** 2)
    try:
        Q = np.linalg.inv(S1) - np.linalg.inv(S0)   # LRT score = x^T Q x
    except np.linalg.LinAlgError:
        return 0.5
    score_real = np.einsum("ij,jk,ik->i", real, Q, real)
    score_scr = np.einsum("ij,jk,ik->i", scr, Q, scr)
    # AUC = P(score_scr > score_real) qua xếp hạng
    all_s = np.concatenate([score_real, score_scr])
    ranks = np.argsort(np.argsort(all_s))
    r_scr = ranks[n:]
    auc = (r_scr.sum() - n * (n - 1) / 2.0) / (n * n)
    return float(max(auc, 1.0 - auc))   # độ chính xác tốt nhất warden đạt được


# =====================================================================================
# 7. ⭐ Q3 — DECOHERENCE-FREE SUBSPACE (DFS) EMBEDDING (robustness có cấu trúc)
# =====================================================================================
#
# Mô hình hoá tấn công kênh (JPEG/resize/nén) như kênh tuyến tính đối xứng C = V diag(s) V^T,
# s_i in [0,1] = hệ số "sống sót" theo hướng v_i. DFS = các hướng s_i≈1 (gần-bất-biến) =>
# nhúng payload vào đó được bảo vệ THỤ ĐỘNG. Vượt C6 (chỉ tăng ECC phản ứng theo QBER).

def symmetric_channel_from_spectrum(d: int, key: bytes, decay: float = 3.0) -> np.ndarray:
    """Sinh kênh đối xứng C=V diag(s) V^T với phổ s suy giảm (mô phỏng low-pass kiểu JPEG)."""
    V = haar_orthogonal(d, key, "dfs-channel-basis")
    idx = np.arange(d)
    s = np.exp(-decay * idx / d)        # s[0]≈1 (bền) -> s[d-1] nhỏ (mất)
    return V @ np.diag(s) @ V.T


def dfs_basis(channel: np.ndarray, k: int) -> Tuple[np.ndarray, np.ndarray]:
    """Trả về (basis d×k các hướng BỀN nhất, hệ số sống sót s_k tương ứng)."""
    w, V = np.linalg.eigh(channel)          # tăng dần
    order = np.argsort(w)[::-1]             # giảm dần theo độ bền
    sel = order[:k]
    return V[:, sel], w[sel]


class DFSPacker:
    """Nhúng/đọc payload bit dọc các hướng DFS (biên độ ±amp). Channel giữ DFS => robust."""

    def __init__(self, basis: np.ndarray, amp: float = 3.0):
        self.basis = basis          # d×k
        self.k = basis.shape[1]
        self.amp = float(amp)

    def embed(self, base_vec: np.ndarray, bits: np.ndarray) -> np.ndarray:
        bits = np.asarray(bits, np.uint8)[: self.k]
        signs = (2.0 * bits - 1.0) * self.amp
        return base_vec + self.basis[:, : bits.size] @ signs

    def extract(self, recv_vec: np.ndarray, survival: np.ndarray, nbits: int) -> np.ndarray:
        coeff = self.basis[:, :nbits].T @ recv_vec
        coeff = coeff / np.maximum(survival[:nbits], 1e-6)   # bù suy giảm theo hướng
        return (coeff > 0).astype(np.uint8)


# =====================================================================================
# 8. ⭐ Q4 — QUANTUM-DIFFUSION SCRAMBLER (chuỗi unitary có khoá, QuDDPM-style; fallback numpy)
# =====================================================================================
#
# Quá trình forward = chuỗi n_steps biến đổi symplectic có khoá (mỗi step phụ thuộc khoá+step).
# Đây là "quantum diffusion" THẬT tác động lên latent (thay vì adapter trơ): "thay nhiễu
# Gaussian bằng unitary". Khả nghịch chính xác (đảo thứ tự step + nghịch từng step).
# qseries_sd.py thay nguồn tham số bằng VQC PennyLane (quantum-generated).

class QuantumDiffusionScrambler:
    def __init__(self, key: bytes, block: int = 64, n_steps: int = 4,
                 squeeze_scale: float = 0.0):
        self.key = key
        self.block = int(block)
        self.n_steps = int(n_steps)
        self.steps: List[SymplecticScrambler] = [
            SymplecticScrambler(derive_subkey(key, f"qddpm-step-{i}"), block=block,
                                squeeze_scale=squeeze_scale)
            for i in range(self.n_steps)
        ]

    def forward(self, x_flat: np.ndarray) -> np.ndarray:
        out = np.asarray(x_flat, np.float64)
        for st in self.steps:
            out = st.forward(out)
        return out

    def inverse(self, y_flat: np.ndarray) -> np.ndarray:
        out = np.asarray(y_flat, np.float64)
        for st in reversed(self.steps):
            out = st.inverse(out)
        return out


# =====================================================================================
# 9. ⭐ Q5 — QUANTUM-MONEY TOKEN (no-cloning; container bất khả giả) — vượt HMAC (C4)
# =====================================================================================
#
# Token = tập "money qubit" (bits+bases dẫn xuất từ K_money + serial). Người có khoá đo đúng
# basis => QBER≈0 (authentic). Kẻ giả KHÔNG biết basis: muốn nhân bản phải đo (intercept-resend)
# ở basis đoán => sụp trạng thái => verifier đo lại sai ~25% => QBER cao => lộ giả mạo.
# (Trên phần cứng cổ điển là MÔ PHỎNG no-cloning; serial bind vào hash container.)

@dataclass
class MoneyToken:
    serial: str
    state_codes: np.ndarray
    container_tag: str   # hmac(K_conf, sha256(container)) — gắn token vào container cụ thể

    def to_dict(self) -> Dict:
        return {"serial": self.serial,
                "state_codes_b64": base64.b64encode(self.state_codes.tobytes()).decode(),
                "n": int(self.state_codes.size), "container_tag": self.container_tag}

    @staticmethod
    def from_dict(dd: Dict) -> "MoneyToken":
        codes = np.frombuffer(base64.b64decode(dd["state_codes_b64"]), dtype=np.uint8)
        return MoneyToken(dd["serial"], codes.copy(), dd["container_tag"])


def mint_quantum_money(qkm: QuantumKeyManager, serial: str, container_path: str,
                       n_money: int = 256) -> MoneyToken:
    bases = rng_from(qkm.K_money(), "money-bases:" + serial).integers(0, 2, n_money, dtype=np.uint8)
    bits = rng_from(qkm.K_money(), "money-bits:" + serial).integers(0, 2, n_money, dtype=np.uint8)
    state_codes = (bits + 2 * bases).astype(np.uint8)
    tag = hmac.new(qkm.K_conf(), hashlib.sha256(Path(container_path).read_bytes()).digest(),
                   hashlib.sha256).hexdigest()
    return MoneyToken(serial, state_codes, tag)


def verify_quantum_money(token: MoneyToken, qkm: QuantumKeyManager, container_path: str,
                         qber_threshold: float = 0.11) -> Dict:
    """Đo token theo basis đúng (từ khoá). Authentic => QBER≈0 và container_tag khớp."""
    n = token.state_codes.size
    bases = rng_from(qkm.K_money(), "money-bases:" + token.serial).integers(0, 2, n, dtype=np.uint8)
    exp_bits = rng_from(qkm.K_money(), "money-bits:" + token.serial).integers(0, 2, n, dtype=np.uint8)
    prep_bits = token.state_codes % 2
    prep_bases = token.state_codes // 2
    # Verifier đo ở basis đúng: nếu token chuẩn bị ở đúng basis -> đọc đúng bit; sai basis -> random
    meas = prep_bits.copy()
    wrong = bases != prep_bases
    if wrong.any():
        meas[wrong] = rng_from(qkm.K_money(), "verify-collapse:" + token.serial
                               ).integers(0, 2, int(wrong.sum()), dtype=np.uint8)
    qber = float(np.mean(meas != exp_bits)) if n else 1.0
    tag = hmac.new(qkm.K_conf(), hashlib.sha256(Path(container_path).read_bytes()).digest(),
                   hashlib.sha256).hexdigest()
    tag_ok = hmac.compare_digest(tag, token.container_tag)
    return {"qber": qber, "authentic": (qber <= qber_threshold) and tag_ok,
            "tag_ok": tag_ok, "n": int(n)}


def forge_quantum_money(token: MoneyToken, seed: int = 0) -> MoneyToken:
    """Kẻ giả KHÔNG có khoá: intercept-resend ở basis ngẫu nhiên -> token sao chép sai."""
    n = token.state_codes.size
    rng = np.random.default_rng(seed)
    guess_bases = rng.integers(0, 2, n, dtype=np.uint8)
    prep_bits = token.state_codes % 2
    prep_bases = token.state_codes // 2
    meas = prep_bits.copy()
    wrong = guess_bases != prep_bases
    meas[wrong] = rng.integers(0, 2, int(wrong.sum()), dtype=np.uint8)   # sụp trạng thái
    forged_codes = (meas + 2 * guess_bases).astype(np.uint8)             # resend ở basis đoán
    return MoneyToken(token.serial, forged_codes, token.container_tag)


# =====================================================================================
# 10. ⭐ Q6 — SWAP-TEST FINGERPRINT (so khớp n-bit với O(log n) "qubit")
# =====================================================================================
#
# Fingerprint nén x in {0,1}^n -> vector chiều d = O(log n) (Johnson-Lindenstrauss). SWAP test:
# P(đo 0) = (1 + |<u|v>|^2)/2. Giống nhau => ~1; khác => thấp. Lợi thế: ĐỘ PHỨC TẠP TRUYỀN
# THÔNG O(log n) thay vì gửi cả n bit / 256-bit hash (advantage CHỨNG MINH ĐƯỢC).

def fingerprint_vector(bits: np.ndarray, d: int, seed: int = 1234) -> np.ndarray:
    """x in {0,1}^n -> u in R^d (chuẩn hoá), d = O(log n) qua phép chiếu ngẫu nhiên cố định."""
    bits = np.asarray(bits, np.float64).reshape(-1)
    n = bits.size
    R = np.random.default_rng(seed).standard_normal((d, n)) / np.sqrt(n)
    v = R @ (2.0 * bits - 1.0)
    nrm = np.linalg.norm(v)
    return v / nrm if nrm > 0 else v


def swap_test_prob(u: np.ndarray, v: np.ndarray) -> float:
    """Xác suất kết quả 0 của SWAP test: P0 = (1 + |<u|v>|^2)/2 in [0.5, 1]."""
    overlap = float(np.dot(u, v)) ** 2
    return 0.5 * (1.0 + overlap)


def fingerprint_dim(n_bits: int, c: float = 8.0) -> int:
    """d = O(log n): chọn d ~ c·log2(n) (>=4)."""
    return max(4, int(np.ceil(c * np.log2(max(2, n_bits)))))


def fingerprint_compare(bitsA: np.ndarray, bitsB: np.ndarray, c: float = 8.0,
                        seed: int = 1234) -> Dict:
    n = max(bitsA.size, bitsB.size)
    d = fingerprint_dim(n, c)
    a = np.zeros(n, np.uint8); a[: bitsA.size] = bitsA
    b = np.zeros(n, np.uint8); b[: bitsB.size] = bitsB
    p0 = swap_test_prob(fingerprint_vector(a, d, seed), fingerprint_vector(b, d, seed))
    return {"p_swap0": p0, "fingerprint_dim": d, "n_bits": int(n),
            "compression": f"{n} bit -> {d} chiu (~{d/max(1,n)*100:.2f}%)"}


# =====================================================================================
# 11. BB84-OTP QUANTUM SEAL (A2 + C2 + C4 + C6) — payload OTP, state_codes%2 KHÔNG lộ bit
# =====================================================================================

@dataclass
class SealResult:
    package_path: str; key_path: str; total_qubits: int; payload_bits: int
    check_bits: int; ecc_repeat: int; sha256_plain: str; container_mac: str


@dataclass
class UnsealResult:
    output_path: str; qber: float; integrity_ok: bool; auth_ok: bool; tampered: bool; note: str = ""


class EavesdroppingDetected(RuntimeError):
    pass


def _bits_from_file(path) -> np.ndarray:
    data = np.fromfile(path, dtype=np.uint8)
    return np.unpackbits(data) if data.size else np.zeros(0, np.uint8)


def _bits_to_file(bits: np.ndarray, path, nbytes: int) -> None:
    if bits.size == 0:
        Path(path).write_bytes(b""); return
    Path(path).write_bytes(np.packbits(bits.astype(np.uint8)).tobytes()[:nbytes])


def _repeat_encode(bits: np.ndarray, r: int) -> np.ndarray:
    return np.repeat(bits, r) if r > 1 else bits


def _repeat_decode(bits: np.ndarray, r: int, n_orig: int) -> np.ndarray:
    if r <= 1:
        return bits[:n_orig]
    usable = (bits.size // r) * r
    maj = bits[:usable].reshape(-1, r).mean(axis=1) >= 0.5
    return maj.astype(np.uint8)[:n_orig]


def seal_encode(input_png, package_path, key_path, qkm: QuantumKeyManager,
                check_rate: float = 0.15, ecc_repeat: int = 1,
                container_path_for_auth: Optional[str] = None) -> SealResult:
    """A2: OTP payload (keystream từ K_enc) TRƯỚC khi tạo state_codes => %2 chỉ lộ ciphertext."""
    payload_bits = _bits_from_file(input_png)
    n_payload = int(payload_bits.size)
    sha_plain = hashlib.sha256(Path(input_png).read_bytes()).hexdigest()

    coded = _repeat_encode(payload_bits, ecc_repeat)
    ks = keystream_bits(qkm.K_enc(), "otp", coded.size)
    cipher = np.bitwise_xor(coded, ks)

    n_cipher = cipher.size
    total = n_cipher if check_rate == 0 else int(np.ceil(n_cipher / (1 - check_rate)))
    n_check = total - n_cipher
    lay = qkm.subkey("seal-layout")
    is_check = np.zeros(total, bool)
    if n_check > 0:
        pos = rng_from(lay, "positions").choice(total, size=n_check, replace=False)
        is_check[pos] = True
    bases = rng_from(lay, "bases").integers(0, 2, total, dtype=np.uint8)
    check_bits = rng_from(lay, "checkbits").integers(0, 2, n_check, dtype=np.uint8)

    bind = keystream_bits(qkm.K_enc(), "otp-bind:" + hashlib.sha3_256(check_bits.tobytes()).hexdigest(),
                          n_cipher)
    cipher = np.bitwise_xor(cipher, bind)

    logical = np.zeros(total, np.uint8)
    logical[~is_check] = cipher
    logical[is_check] = check_bits
    state_codes = (logical + 2 * bases).astype(np.uint8)

    Path(package_path).parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(package_path, state_codes=state_codes)

    auth_src = container_path_for_auth or input_png
    cmac = hmac.new(qkm.K_conf(), hashlib.sha256(Path(auth_src).read_bytes()).digest(),
                    hashlib.sha256).hexdigest()
    meta = {"scheme": "qseries_bb84_otp_v1", "protocol": qkm.protocol,
            "original_filename": Path(input_png).name,
            "original_size_bytes": int(os.path.getsize(input_png)),
            "payload_bits": n_payload, "ecc_repeat": int(ecc_repeat), "total_qubits": int(total),
            "check_bits": int(n_check), "check_rate": float(check_rate),
            "salt_b64": base64.b64encode(qkm.salt).decode(),
            "sha256_plain": sha_plain, "container_mac": cmac,
            "note": "Payload OTP-encrypted; state_codes%2 chi lo CIPHERTEXT."}
    Path(key_path).write_text(json.dumps(meta, indent=2), encoding="utf-8")
    return SealResult(str(package_path), str(key_path), int(total), n_payload,
                      int(n_check), int(ecc_repeat), sha_plain, cmac)


def seal_decode(package_path, key_path, output_png, qkm: QuantumKeyManager,
                qber_threshold: float = 0.11, gate_on_tamper: bool = True,
                container_path_for_auth: Optional[str] = None) -> UnsealResult:
    state_codes = np.load(package_path)["state_codes"].astype(np.uint8)
    meta = json.loads(Path(key_path).read_text(encoding="utf-8"))
    total = int(meta["total_qubits"]); n_payload = int(meta["payload_bits"])
    ecc_repeat = int(meta["ecc_repeat"]); nbytes = int(meta["original_size_bytes"])

    lay = qkm.subkey("seal-layout")
    is_check = np.zeros(total, bool)
    n_check = int(meta["check_bits"])
    if n_check > 0:
        pos = rng_from(lay, "positions").choice(total, size=n_check, replace=False)
        is_check[pos] = True
    expected_check = rng_from(lay, "checkbits").integers(0, 2, n_check, dtype=np.uint8)

    measured = (state_codes % 2).astype(np.uint8)
    observed_check = measured[is_check]
    qber = float(np.mean(observed_check != expected_check)) if n_check else 0.0
    tampered = qber > qber_threshold
    if tampered and gate_on_tamper:
        raise EavesdroppingDetected(f"QBER={qber:.4f} > {qber_threshold} => goi luong tu bi nhieu loan.")

    cipher = measured[~is_check]
    bind = keystream_bits(qkm.K_enc(), "otp-bind:" + hashlib.sha3_256(observed_check.tobytes()).hexdigest(),
                          cipher.size)
    cipher = np.bitwise_xor(cipher, bind)
    ks = keystream_bits(qkm.K_enc(), "otp", cipher.size)
    coded = np.bitwise_xor(cipher, ks)
    payload = _repeat_decode(coded, ecc_repeat, n_payload)

    _bits_to_file(payload, output_png, nbytes)
    sha_now = hashlib.sha256(Path(output_png).read_bytes()).hexdigest()
    integrity_ok = (sha_now == meta["sha256_plain"])
    auth_ok = True
    if container_path_for_auth is not None:
        cmac = hmac.new(qkm.K_conf(), hashlib.sha256(Path(container_path_for_auth).read_bytes()).digest(),
                        hashlib.sha256).hexdigest()
        auth_ok = hmac.compare_digest(cmac, meta.get("container_mac", ""))
    note = "" if integrity_ok else "Integrity mismatch (passphrase sai hoac goi bi sua)."
    return UnsealResult(str(output_png), qber, integrity_ok, auth_ok, tampered, note)


def simulate_eavesdropper_on_package(package_path, interception_rate: float = 1.0,
                                     seed: int = 0) -> Dict:
    state_codes = np.load(package_path)["state_codes"].astype(np.uint8)
    total = state_codes.size
    rng = np.random.default_rng(seed)
    prep_bits = state_codes % 2; prep_bases = state_codes // 2
    eve_bases = rng.integers(0, 2, total, dtype=np.uint8)
    mask = rng.random(total) < interception_rate
    wrong = mask & (eve_bases != prep_bases)
    new_bits = prep_bits.copy()
    new_bits[wrong] = rng.integers(0, 2, int(wrong.sum()), dtype=np.uint8)
    new_codes = (new_bits + 2 * np.where(mask, eve_bases, prep_bases)).astype(np.uint8)
    np.savez_compressed(package_path, state_codes=new_codes)
    return {"intercepted": int(mask.sum()), "total": int(total)}


# =====================================================================================
# SELF-TEST (chạy: python qseries_core.py)
# =====================================================================================

def _selftest() -> bool:
    import tempfile
    from scipy import stats
    rng = np.random.default_rng(0)
    ok: List[bool] = []

    def check(name, cond):
        ok.append(bool(cond))
        print(f"  [{'PASS' if cond else 'FAIL'}] {name}")

    print("=" * 74); print("SELF-TEST qseries_core (Q-series)"); print("=" * 74)

    salt = b"\x01" * 16

    # --- Hạ tầng: QKD + đa khoá ---
    print("[ Hạ tầng QKD / đa khoá ]")
    k_h, rep_h = bb84_simulate("pw", salt, eve_rate=0.0)
    _, rep_e = bb84_simulate("pw", salt, eve_rate=1.0)
    check("BB84 honest QBER<=0.02", rep_h.qber <= 0.02)
    check("BB84 Eve QBER>0.15", rep_e.qber > 0.15)
    _, e_h = e91_simulate("pw", salt, eve_rate=0.0)
    _, e_e = e91_simulate("pw", salt, eve_rate=1.0)
    check(f"E91 honest |S|>2.6 (got {abs(e_h.chsh_S):.3f})", abs(e_h.chsh_S) > 2.6)
    check(f"E91 Eve |S|<2.4 (got {abs(e_e.chsh_S):.3f})", abs(e_e.chsh_S) < 2.4)
    qkm = QuantumKeyManager("secret-pass", protocol="bb84", salt=salt); qkm.establish()
    subs = {qkm.K_seed(), qkm.K_qw(), qkm.K_enc(), qkm.K_conf(), qkm.K_money(), qkm.K_dfs()}
    check("B3 6 subkey phân biệt", len(subs) == 6)

    # --- C7 quantum-walk permutation ---
    print("[ C7 quantum-walk permutation ]")
    perm = quantum_walk_permutation(qkm.K_qw(), 512)
    check("C7 perm hợp lệ", sorted(perm.tolist()) == list(range(512)))
    inv = invert_permutation(perm); x = rng.standard_normal(512)
    check("C7 perm khả nghịch", np.allclose(apply_permutation(apply_permutation(x, perm), inv), x))

    # --- Q1 SymplecticScrambler ---
    print("[ Q1 — CV-Symplectic Scrambler ]")
    z = rng.standard_normal(4 * 64 * 64)
    sp_passive = SymplecticScrambler(qkm.K_seed(), block=64, squeeze_scale=0.0)
    y0 = sp_passive.forward(z)
    check("Q1 passive round-trip exact", np.allclose(sp_passive.inverse(y0), z, atol=1e-9))
    check("Q1 passive bảo toàn norm (orthogonal)", abs(np.linalg.norm(y0) - np.linalg.norm(z)) < 1e-6)
    ksp = stats.kstest((y0 - y0.mean()) / y0.std(), "norm").pvalue
    check(f"Q1 passive bảo toàn Gaussian (KS p={ksp:.3f}>0.01)", ksp > 0.01)
    sp_active = SymplecticScrambler(qkm.K_seed(), block=64, squeeze_scale=0.5)
    ya = sp_active.forward(z)
    check("Q1 active round-trip exact (vẫn khả nghịch)", np.allclose(sp_active.inverse(ya), z, atol=1e-9))
    check("Q1 active đổi norm (squeeze hoạt động)", abs(np.linalg.norm(ya) - np.linalg.norm(z)) > 1.0)
    sp_other = SymplecticScrambler(derive_subkey(qkm.sifted_key, "x"), block=64)
    check("Q1 sai khoá => không khôi phục", not np.allclose(sp_other.inverse(y0), z, atol=1e-3))

    # --- Q1-spatial ChannelMixingScrambler: round-trip + GIỮ cấu trúc không gian ---
    print("[ Q1-spatial — Channel-Mixing Scrambler (container tự nhiên) ]")
    cm = ChannelMixingScrambler(qkm.K_seed(), channels=4)
    xchw = rng.standard_normal((4, 16, 16))
    ycm = cm.forward(xchw)
    check("Q1-cm round-trip exact", np.allclose(cm.inverse(ycm), xchw, atol=1e-9))
    check("Q1-cm bảo toàn norm", abs(np.linalg.norm(ycm) - np.linalg.norm(xchw)) < 1e-9)
    gyy, gxx = np.meshgrid(np.linspace(0, 3, 16), np.linspace(0, 3, 16), indexing="ij")
    smooth = np.stack([np.sin(gxx + c) + np.cos(gyy - c) for c in range(4)])   # latent TRƠN
    def _tv(a):   # total variation không gian (độ "gồ ghề")
        return float(np.abs(np.diff(a, axis=1)).sum() + np.abs(np.diff(a, axis=2)).sum())
    tv0 = _tv(smooth)
    tv_cm = _tv(cm.forward(smooth))
    perm_full = rng_from(qkm.K_seed(), "cmp-perm").permutation(smooth.size)
    tv_perm = _tv(smooth.reshape(-1)[perm_full].reshape(smooth.shape))
    check(f"Q1-cm GIỮ cấu trúc không gian (TV x{tv_cm/tv0:.2f}) << permutation (TV x{tv_perm/tv0:.1f})",
          tv_cm < 3.0 * tv0 and tv_cm < 0.3 * tv_perm)
    cm2 = ChannelMixingScrambler(derive_subkey(qkm.sifted_key, "cm2"), channels=4)
    check("Q1-cm key-sensitive", not np.allclose(cm2.forward(xchw), ycm, atol=1e-3))
    rep_cm = helstrom_undetectability(np.eye(4), cm.block_covariance(1.0))
    check("Q1-cm Helstrom passive (warden 1/2)",
          rep_cm.is_passive and abs(rep_cm.p_detect_upper - 0.5) < 1e-9)

    # --- Q2 Helstrom ---
    print("[ Q2 — Helstrom-bounded undetectability ]")
    S0 = np.eye(64)
    rep_passive = helstrom_undetectability(S0, sp_passive.block_covariance(1.0))
    check("Q2 passive: warden ở mức 1/2 (p_detect=0.5)", abs(rep_passive.p_detect_upper - 0.5) < 1e-6)
    check("Q2 passive: is_passive=True", rep_passive.is_passive)
    rep_active = helstrom_undetectability(S0, sp_active.block_covariance(1.0))
    check(f"Q2 active: advantage>0 (adv_upper={rep_active.advantage_upper:.3f})",
          rep_active.advantage_upper > 0.0)
    check("Q2 active: p_detect>0.5", rep_active.p_detect_upper > 0.5)
    auc_p = empirical_detect_auc(sp_passive, n=3000)
    check(f"Q2 thực nghiệm passive: warden≈mù ({auc_p:.3f}≈0.5)", auc_p < 0.56)
    auc_a = empirical_detect_auc(sp_active, n=3000)
    check(f"Q2 thực nghiệm active: warden phân biệt được ({auc_a:.3f}>0.55)", auc_a > 0.55)

    # --- Q3 DFS ---
    print("[ Q3 — Decoherence-Free Subspace embedding ]")
    d = 128
    C = symmetric_channel_from_spectrum(d, qkm.K_dfs(), decay=4.0)
    basis_robust, surv_robust = dfs_basis(C, k=16)
    check("Q3 DFS hệ số sống sót cao (>0.5)", float(surv_robust.min()) > 0.5)
    # so BER: nhúng dọc DFS bền vs dọc hướng yếu nhất
    w_all, V_all = np.linalg.eigh(C)
    weak_basis = V_all[:, np.argsort(w_all)[:16]]
    weak_surv = np.sort(w_all)[:16]
    bits = rng.integers(0, 2, 16, dtype=np.uint8)
    base = rng.standard_normal(d) * 0.1
    noise = lambda: rng.standard_normal(d) * 0.25
    pk_r = DFSPacker(basis_robust, amp=3.0)
    recv_r = C @ pk_r.embed(base, bits) + noise()
    ber_r = float(np.mean(pk_r.extract(recv_r, surv_robust, 16) != bits))
    pk_w = DFSPacker(weak_basis, amp=3.0)
    recv_w = C @ pk_w.embed(base, bits) + noise()
    ber_w = float(np.mean(pk_w.extract(recv_w, weak_surv, 16) != bits))
    check(f"Q3 BER(DFS bền)={ber_r:.2f} < BER(hướng yếu)={ber_w:.2f}", ber_r < ber_w)
    check("Q3 BER(DFS bền) thấp (<0.1)", ber_r < 0.1)

    # --- Q4 Quantum-Diffusion Scrambler ---
    print("[ Q4 — Quantum-Diffusion Scrambler (chuỗi unitary có khoá) ]")
    qd = QuantumDiffusionScrambler(qkm.K_seed(), block=64, n_steps=5, squeeze_scale=0.0)
    yq = qd.forward(z)
    check("Q4 round-trip exact qua n_steps", np.allclose(qd.inverse(yq), z, atol=1e-8))
    check("Q4 passive bảo toàn norm", abs(np.linalg.norm(yq) - np.linalg.norm(z)) < 1e-5)
    qd2 = QuantumDiffusionScrambler(derive_subkey(qkm.sifted_key, "y"), block=64, n_steps=5)
    check("Q4 key-sensitive (sai khoá => khác)", not np.allclose(qd2.forward(z), yq, atol=1e-3))

    # --- Q5 Quantum money ---
    print("[ Q5 — Quantum-money token ]")
    with tempfile.TemporaryDirectory() as dtmp:
        dtmp = Path(dtmp)
        cont = dtmp / "hide.png"; cont.write_bytes(rng.integers(0, 256, 800, dtype=np.uint8).tobytes())
        qkm_m = QuantumKeyManager("pp", salt=salt); qkm_m.establish()
        tok = mint_quantum_money(qkm_m, "SN-0001", str(cont), n_money=512)
        v_ok = verify_quantum_money(tok, qkm_m, str(cont))
        check(f"Q5 honest verify authentic (qber={v_ok['qber']:.3f})", v_ok["authentic"])
        forged = forge_quantum_money(tok, seed=7)
        v_forge = verify_quantum_money(forged, qkm_m, str(cont))
        check(f"Q5 forged bị từ chối (qber={v_forge['qber']:.3f}>0.11)", not v_forge["authentic"])
        # token gắn container: đổi container => tag fail
        fake = dtmp / "fake.png"; fake.write_bytes(b"x" * 800)
        v_fake = verify_quantum_money(tok, qkm_m, str(fake))
        check("Q5 token gắn container (đổi container => from chối)", not v_fake["authentic"])

    # --- Q6 SWAP fingerprint ---
    print("[ Q6 — SWAP-test fingerprint ]")
    nbits = 8192
    bA = rng.integers(0, 2, nbits, dtype=np.uint8)
    same = fingerprint_compare(bA, bA.copy())
    check(f"Q6 giống nhau P(swap0)≈1 (got {same['p_swap0']:.3f})", same["p_swap0"] > 0.98)
    bB = bA.copy(); flip = rng.choice(nbits, nbits // 2, replace=False); bB[flip] ^= 1
    diff = fingerprint_compare(bA, bB)
    check(f"Q6 khác nhiều P(swap0) thấp (got {diff['p_swap0']:.3f}<0.9)", diff["p_swap0"] < 0.9)
    check(f"Q6 nén n->O(log n): {same['compression']}", same["fingerprint_dim"] < nbits // 50)

    # --- A2/C2/C4 seal ---
    print("[ A2/C2/C4 — BB84-OTP seal ]")
    with tempfile.TemporaryDirectory() as dtmp:
        dtmp = Path(dtmp)
        secret = rng.integers(0, 256, 1500, dtype=np.uint8).tobytes()
        png = dtmp / "hide.png"; png.write_bytes(secret)
        qkm2 = QuantumKeyManager("pp", salt=salt); qkm2.establish()
        seal_encode(png, dtmp / "pkg.npz", dtmp / "key.json", qkm2, check_rate=0.15,
                    ecc_repeat=3, container_path_for_auth=str(png))
        codes = np.load(dtmp / "pkg.npz")["state_codes"]
        leaked = np.packbits((codes % 2)[: len(secret) * 8]).tobytes()[: len(secret)]
        frac = np.mean(np.frombuffer(leaked, np.uint8) == np.frombuffer(secret, np.uint8))
        check(f"A2 payload KHÔNG lộ qua state_codes%2 (match={frac:.3f}~rand)", frac < 0.6)
        out = seal_decode(dtmp / "pkg.npz", dtmp / "key.json", dtmp / "rev.png", qkm2,
                          container_path_for_auth=str(png))
        check("A2 honest decode integrity_ok", out.integrity_ok)
        check("C4 container auth_ok", out.auth_ok)
        seal_encode(png, dtmp / "pkg2.npz", dtmp / "key2.json", qkm2, check_rate=0.15, ecc_repeat=3,
                    container_path_for_auth=str(png))
        simulate_eavesdropper_on_package(dtmp / "pkg2.npz", 1.0)
        det = False
        try:
            o2 = seal_decode(dtmp / "pkg2.npz", dtmp / "key2.json", dtmp / "rev2.png", qkm2,
                             container_path_for_auth=str(png))
            det = (not o2.integrity_ok) or o2.tampered
        except EavesdroppingDetected:
            det = True
        check("C2 tamper bị phát hiện", det)

    print("=" * 74)
    print(f"KẾT QUẢ: {sum(ok)}/{len(ok)} PASS")
    print("=" * 74)
    return all(ok)


In [ ]:
"""
qseries_sd.py
=============
Tầng tích hợp **QuStega Q-series** vào pipeline Stable Diffusion / CRoSS (torch + diffusers,
chạy trên Kaggle GPU). Phần logic quantum/crypto đã được kiểm chứng số trong
qseries_core._selftest() (35/35 PASS). File này chỉ ghép chúng vào luồng sinh ảnh.

Tích hợp Q1-Q6 vào một pipeline duy nhất `run_inference_qseries`:
  Q1  CV-Symplectic Scrambler  : scramble latent xT (passive=vô hình; squeeze=núm đánh đổi).
  Q2  Helstrom report          : in chặn bất-khả-phát-hiện của warden cho cấu hình đang dùng.
  Q3  DFS robustness probe      : đo không-gian-con latent bền JPEG (hướng đặt payload thụ động).
  Q4  Quantum-Diffusion gen     : tham số biến đổi latent SINH TỪ VQC PennyLane (quantum thật),
                                  fallback khoá cổ điển nếu thiếu PennyLane.
  Q5  Quantum-money token       : token bất khả giả gắn vào hide.png (no-cloning, vượt HMAC).
  Q6  SWAP fingerprint          : so khớp hide gốc vs hide khôi phục với O(log n) "qubit".

Trong NOTEBOOK Kaggle: cell `qseries_core` chạy TRƯỚC nên mọi tên đã ở global namespace,
`from qseries_core import *` sẽ ImportError và bị bỏ qua (vô hại). Khi chạy như module/local
thì import bình thường.
"""
from __future__ import annotations

import os
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
from PIL import Image

try:  # trong notebook Kaggle: bỏ qua (tên đã có ở global từ cell core). Local: import thật.
    from qseries_core import *          # noqa: F401,F403
    from qseries_core import (
        QuantumKeyManager, SymplecticScrambler, QuantumDiffusionScrambler, ChannelMixingScrambler,
        quantum_walk_permutation, invert_permutation, derive_subkey, rng_from, shake,
        seal_encode, seal_decode, simulate_eavesdropper_on_package, EavesdroppingDetected,
        helstrom_undetectability, mint_quantum_money, verify_quantum_money, forge_quantum_money,
        fingerprint_compare, DeniableMultiplexer,
    )
except ImportError:
    pass

try:
    import torch
    HAS_TORCH = True
except Exception:
    torch = None
    HAS_TORCH = False

try:
    import pennylane as qml
    HAS_PENNYLANE = True
except Exception:
    qml = None
    HAS_PENNYLANE = False

try:
    import cv2
    HAS_CV2 = True
except Exception:
    cv2 = None
    HAS_CV2 = False


# =====================================================================================
# Q4 — QUANTUM-GENERATED KEY: tham số biến đổi latent dẫn xuất từ ĐO LƯỜNG mạch lượng tử
# =====================================================================================

def quantum_derived_key(base_key: bytes, n_qubits: int = 6, depth: int = 2,
                        n_runs: int = 6) -> Tuple[bytes, str]:
    """
    Q4/C5 — chạy một VQC (AngleEmbedding + StronglyEntanglingLayers) với input/weights dẫn
    xuất từ base_key; băm các giá trị kỳ vọng PauliZ đo được thành một khoá 32 byte. => toàn bộ
    chuỗi unitary của Quantum-Diffusion Scrambler được "sinh bởi mạch lượng tử". Không có
    PennyLane thì trả lại base_key (fallback cổ điển) — API & tính khả nghịch không đổi.
    """
    if not HAS_PENNYLANE:
        return base_key, "classical-fallback"
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev)
    def circ(inputs, weights):
        qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="Y")
        qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    meas = []
    for r in range(n_runs):
        rng = rng_from(base_key, f"vqc-input-{r}")
        inputs = rng.uniform(-np.pi, np.pi, size=n_qubits)
        weights = rng_from(base_key, f"vqc-weight-{r}").uniform(-np.pi, np.pi, size=(depth, n_qubits, 3))
        vals = np.asarray(circ(inputs, weights), dtype=np.float64)
        meas.append(vals)
    meas_bytes = np.concatenate(meas).astype(np.float64).tobytes()
    qkey = shake(meas_bytes, "quantum-derived-key", 32)
    return qkey, f"pennylane-vqc(n_qubits={n_qubits},depth={depth})"


# =====================================================================================
# LATENT KEY TRANSFORM — C7 permutation ∘ (Q1 symplectic | Q4 quantum-diffusion)
# =====================================================================================

class LatentKeyTransformQ:
    """
    Biến đổi khoá hoá khả nghịch trên latent — bản **CHANNEL-MIXING (spatial-coherent)**:
    trộn KÊNH bằng orthogonal C×C áp ĐỒNG NHẤT mọi vị trí (h,w) => **GIỮ cấu trúc không gian**
    => container TỰ NHIÊN. (Thay cho bản cũ permute-toàn-cục + block-scramble vốn phá bố cục
    khiến hide kỳ lạ VÀ làm VAE round-trip kém => reverse mờ.) Vẫn orthogonal nên Q1/Q2 giữ
    nguyên: khả nghịch CHÍNH XÁC + Helstrom passive. Khoá Qc = quantum_derived_key (Q4: VQC).
    """

    def __init__(self, qkm: "QuantumKeyManager", latent_shape, channels: int = None):
        self.latent_shape = tuple(int(s) for s in latent_shape)
        c = channels if channels is not None else self.latent_shape[-3]   # (B,C,H,W) -> C
        qkey, self.qmode = quantum_derived_key(qkm.K_seed())   # Q4
        self.scrambler = ChannelMixingScrambler(qkey, channels=c)
        self.channels = c
        self.kind = f"Q1-channel-mixing(C={c}, spatial-coherent)"

    def _np(self, xT):
        return xT.detach().cpu().to(torch.float64).numpy(), xT.dtype, xT.device

    def hide(self, xT):
        arr, dt, dev = self._np(xT)
        return torch.from_numpy(self.scrambler.forward(arr)).to(device=dev, dtype=dt)

    def reveal(self, xT_scr):
        arr, dt, dev = self._np(xT_scr)
        return torch.from_numpy(self.scrambler.inverse(arr)).to(device=dev, dtype=dt)

    def helstrom_report(self, sigma2: float = 1.0):
        """Q2 — biên bất-khả-phát-hiện trên KHÔNG GIAN KÊNH (Qc Qc^T = I => passive)."""
        return helstrom_undetectability(np.eye(self.channels), self.scrambler.block_covariance(sigma2))


# =====================================================================================
# Q3 — DFS PROBE: đo không-gian-con latent BỀN dưới JPEG (hướng đặt payload thụ động)
# =====================================================================================

def jpeg_roundtrip(rgb_uint8: np.ndarray, quality: int = 75) -> np.ndarray:
    """Nén/giải JPEG một ảnh RGB uint8 (mô phỏng kênh mạng xã hội)."""
    import io
    buf = io.BytesIO()
    Image.fromarray(rgb_uint8).save(buf, format="JPEG", quality=quality)
    buf.seek(0)
    return np.array(Image.open(buf).convert("RGB"))


def dfs_robustness_probe(ode, img_rgb: np.ndarray, quality: int = 75) -> dict:
    """
    Q3 — đo độ "sống sót" của latent dưới JPEG: encode latent của ảnh gốc vs ảnh-sau-JPEG;
    báo cáo tỉ lệ năng lượng giữ lại theo kênh latent (hướng càng giữ nhiều = DFS càng bền).
    Đây là chẩn đoán để biết nên đặt payload/secret vào đâu cho robust (placement thụ động).
    """
    z0 = ode.image2latent(img_rgb).detach().cpu().numpy().reshape(-1)
    z1 = ode.image2latent(jpeg_roundtrip(img_rgb, quality)).detach().cpu().numpy().reshape(-1)
    num = float(np.sum(z0 * z1))
    den = float(np.linalg.norm(z0) * np.linalg.norm(z1) + 1e-9)
    cos_sim = num / den
    rel_err = float(np.linalg.norm(z1 - z0) / (np.linalg.norm(z0) + 1e-9))
    surviving_frac = float(np.mean(np.abs(z1 - z0) < 0.5 * np.std(z0)))
    return {"jpeg_quality": quality, "latent_cos_sim": round(cos_sim, 4),
            "latent_rel_err": round(rel_err, 4), "dfs_surviving_frac": round(surviving_frac, 4)}


# =====================================================================================
# ODE SOLVE — CRoSS DDIM inversion (mặc định) + EDICT (tuỳ chọn), guidance_scale=1.0
# =====================================================================================

class ODESolveQ:
    def __init__(self, model, num_ddim_steps=25, inverter="dpm++", edict_p=0.93, guidance_scale=1.0):
        self.model = model
        self.num_ddim_steps = num_ddim_steps
        self.tokenizer = model.tokenizer
        self.context = None
        self.inverter = inverter
        self.edict_p = edict_p
        self.guidance_scale = guidance_scale
        if inverter in ("ddim", "edict"):
            self.model.scheduler.set_timesteps(num_ddim_steps)
        else:
            self._init_dpm_schedulers(num_ddim_steps)

    @property
    def scheduler(self):
        return self.model.scheduler

    def _alpha(self, t):
        return self.scheduler.alphas_cumprod[t] if t >= 0 else self.scheduler.final_alpha_cumprod

    def _init_dpm_schedulers(self, num_steps: int):
        """Q-DPM: khởi tạo cặp DPM-Solver++ (generation + inversion) độc lập với pipe.scheduler."""
        _cfg = dict(
            beta_start=0.00085, beta_end=0.012, beta_schedule="scaled_linear",
            solver_order=2, algorithm_type="dpmsolver++",
            thresholding=False,
        )
        self._dpm_gen = DPMSolverMultistepScheduler(**_cfg)          # noise→image
        self._dpm_inv = DPMSolverMultistepInverseScheduler(**_cfg)   # image→noise
        self._dpm_steps = num_steps

    def _get_eps(self, latents, t, context=None):
        """CFG noise prediction, guidance_scale=1.0 (bắt buộc cho inversion chính xác)."""
        context = context if context is not None else self.context
        uncond, cond = context.chunk(2)
        nu = self.model.unet(latents, t, uncond)["sample"]
        nt = self.model.unet(latents, t, cond)["sample"]
        return nu + self.guidance_scale * (nt - nu)

    def prev_step(self, eps, t, sample):
        prev_t = t - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        a_t, a_p = self.scheduler.alphas_cumprod[t], self._alpha(prev_t)
        x0 = (sample - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5
        return a_p ** 0.5 * x0 + (1 - a_p) ** 0.5 * eps

    def next_step(self, eps, t, sample):
        t, nt = min(t - self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps, 999), t
        a_t, a_n = self._alpha(t), self.scheduler.alphas_cumprod[nt]
        x0 = (sample - (1 - a_t) ** 0.5 * eps) / a_t ** 0.5
        return a_n ** 0.5 * x0 + (1 - a_n) ** 0.5 * eps

    def get_noise_pred(self, latents, t, is_forward=True, context=None):
        """DDIM/EDICT noise step (giữ cho backward compatibility)."""
        eps = self._get_eps(latents, t, context)
        return self.next_step(eps, t, latents) if is_forward else self.prev_step(eps, t, latents)

    @torch.no_grad()
    def latent2image(self, latents, return_type="np"):
        latents = 1 / 0.18215 * latents.detach()
        image = self.model.vae.decode(latents)["sample"]
        if return_type == "np":
            image = (image / 2 + 0.5).clamp(0, 1)
            image = (image.cpu().permute(0, 2, 3, 1).numpy()[0] * 255).astype(np.uint8)
        return image

    @torch.no_grad()
    def image2latent(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)
        if isinstance(image, torch.Tensor) and image.dim() == 4:
            return image
        image = torch.from_numpy(image).float() / 127.5 - 1
        image = image.permute(2, 0, 1).unsqueeze(0).to(self.model.device)
        return self.model.vae.encode(image)["latent_dist"].mean * 0.18215

    @torch.no_grad()
    def init_prompt(self, prompt: str):
        tok = self.model.tokenizer
        u = self.model.text_encoder(tok([""], padding="max_length",
            max_length=tok.model_max_length, return_tensors="pt").input_ids.to(self.model.device))[0]
        t = self.model.text_encoder(tok([prompt], padding="max_length",
            max_length=tok.model_max_length, truncation=True,
            return_tensors="pt").input_ids.to(self.model.device))[0]
        self.context = torch.cat([u, t])

    @torch.no_grad()
    def ddim_loop(self, latent, is_forward=True):
        latent = latent.clone().detach()
        for i in range(self.num_ddim_steps):
            t = (self.scheduler.timesteps[len(self.scheduler.timesteps) - i - 1]
                 if is_forward else self.scheduler.timesteps[i])
            latent = self.get_noise_pred(latent, t, is_forward, self.context)
        return latent

    @torch.no_grad()
    def dpm_loop(self, latent, is_forward=True):
        """
        DPM-Solver++ bậc 2 — vượt DDIM bậc 1 về độ chính xác inversion.
        is_forward=True  → inversion  (image→noise): DPMSolverMultistepInverseScheduler
        is_forward=False → generation (noise→image): DPMSolverMultistepScheduler
        Lợi thế: 25 bước đạt chất lượng bằng DDIM 50 bước, sai số inversion thấp hơn.
        """
        latent = latent.clone().detach()
        sched = self._dpm_inv if is_forward else self._dpm_gen
        sched.set_timesteps(self._dpm_steps, device=self.model.device)
        for t in sched.timesteps:
            eps = self._get_eps(latent, t)
            latent = sched.step(eps, t, latent).prev_sample
        return latent

    @torch.no_grad()
    def _edict_coef(self, t):
        step = self.scheduler.config.num_train_timesteps // self.scheduler.num_inference_steps
        s = t - step
        a_t, a_s = self.scheduler.alphas_cumprod[t], self._alpha(s)
        a = (a_s / a_t) ** 0.5
        b = (1 - a_s) ** 0.5 - (a_s * (1 - a_t) / a_t) ** 0.5
        return a, b

    @torch.no_grad()
    def edict_loop(self, latent, is_forward=True):
        p = self.edict_p
        x = latent.clone(); y = latent.clone()
        uncond, cond = self.context.chunk(2)

        def eps(z, t):
            nu = self.model.unet(z, t, uncond)["sample"]
            nt = self.model.unet(z, t, cond)["sample"]
            return nu + self.guidance_scale * (nt - nu)

        for i in range(self.num_ddim_steps):
            t = (self.scheduler.timesteps[len(self.scheduler.timesteps) - i - 1]
                 if is_forward else self.scheduler.timesteps[i])
            a, b = self._edict_coef(int(t))
            if is_forward:
                y_int = (y - (1 - p) * x) / p
                x_int = (x - (1 - p) * y_int) / p
                y = (y_int - b * eps(x_int, t)) / a
                x = (x_int - b * eps(y, t)) / a
            else:
                x_int = a * x + b * eps(y, t)
                y_int = a * y + b * eps(x_int, t)
                x = p * x_int + (1 - p) * y_int
                y = p * y_int + (1 - p) * x
        return (x + y) / 2

    def loop(self, latent, is_forward=True):
        if self.inverter == "dpm++":
            return self.dpm_loop(latent, is_forward)
        elif self.inverter == "edict":
            return self.edict_loop(latent, is_forward)
        else:
            return self.ddim_loop(latent, is_forward)

    def invert(self, prompt, start_latent, is_forward):
        self.init_prompt(prompt)
        return self.loop(start_latent, is_forward=is_forward)


# =====================================================================================
# B2 — RECEIVER RESTORE (khử nhiễu kênh trước inversion)
# =====================================================================================

def receiver_restore(image_rgb_uint8: np.ndarray, strength: int = 3) -> np.ndarray:
    if HAS_CV2:
        return cv2.fastNlMeansDenoisingColored(image_rgb_uint8, None, strength, strength, 7, 21)
    from scipy.ndimage import gaussian_filter
    out = np.stack([gaussian_filter(image_rgb_uint8[..., c], sigma=0.6) for c in range(3)], axis=-1)
    return out.astype(np.uint8)


def _save_rgb(path, rgb_uint8):
    if HAS_CV2:
        cv2.imwrite(path, cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2BGR))
    else:
        Image.fromarray(rgb_uint8).save(path)


def _load_rgb(path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"))


# =====================================================================================
# ⭐ PIPELINE HỢP NHẤT Q-SERIES (Q1..Q6 trong một luồng)
# =====================================================================================

def run_inference_qseries(
    pipe, image_path, private_key="Eiffel tower", public_key="a tree",
    save_path="/kaggle/working/output_dpm", num_steps=25, passphrase="qseries-secret",
    protocol="bb84", inverter="dpm++",
    scramble_block=64, squeeze_scale=0.0,        # Q1: squeeze_scale=0 => passive (vô hình, provable)
    use_qddpm=True, qddpm_steps=4,               # Q4: chuỗi unitary sinh từ VQC
    eve_rate_key=0.0,                            # nghe lén KÊNH KHOÁ -> gate trước khi hide (C2/C4)
    simulate_eve_package=False, eve_interception_rate=1.0,  # nghe lén GÓI -> tamper-evidence
    jpeg_probe_quality=75,                       # Q3: chất lượng JPEG để dò DFS
    money_serial="QSTEGA-0001", n_money=512,     # Q5
    decoy_image_path=None,                       # C3 (tuỳ chọn)
    device=None, verbose=True,
):
    device = device or pipe.device
    os.makedirs(save_path, exist_ok=True)
    qdir = os.path.join(save_path, "quantum_wrap"); os.makedirs(qdir, exist_ok=True)
    log = (lambda *a: print(*a)) if verbose else (lambda *a: None)

    # --- 1) QKD: thiết lập khoá + gate kênh (C2/C4/C5/B3/C6) ---
    qkm = QuantumKeyManager(passphrase, protocol=protocol, eve_rate=eve_rate_key, salt=os.urandom(16))
    rep = qkm.establish()
    log(f"[QKD] protocol={rep.protocol} qber={rep.qber:.4f} chsh_S={rep.chsh_S:.3f} secure={rep.secure}")
    if not rep.secure:
        raise EavesdroppingDetected(f"Kênh khoá nghi nghe lén: {rep.note} -> HUỶ giấu tin.")
    adapt = qkm.adapt()
    num_steps = max(num_steps, adapt["num_steps"]) if eve_rate_key > 0 else num_steps

    ode = ODESolveQ(pipe, num_ddim_steps=num_steps, inverter=inverter)

    # --- 2) Ảnh & latent + Q3 DFS probe ---
    img = np.array(Image.open(image_path).convert("RGB").resize((512, 512)))
    z0 = ode.image2latent(img)
    _save_rgb(os.path.join(save_path, "gt.png"), img)
    dfs = dfs_robustness_probe(ode, img, quality=jpeg_probe_quality)
    log(f"[Q3-DFS] {dfs}")

    # --- 3) Biến đổi khoá hoá latent (C7 ∘ Q1/Q4) + Q2 Helstrom report ---
    lkt = LatentKeyTransformQ(qkm, latent_shape=tuple(z0.shape))   # Q1 channel-mixing (spatial-coherent)
    log(f"[Q1/Q4] latent transform = {lkt.kind} | quantum-key mode = {lkt.qmode}")
    hrep = lkt.helstrom_report()
    log(f"[Q2-Helstrom] passive={hrep.is_passive} p_detect_upper={hrep.p_detect_upper:.4f} "
        f"advantage_upper={hrep.advantage_upper:.4f} :: {hrep.note}")

    # --- 4) HIDE: forward(private) -> permute+scramble -> [deniability] -> backward(public) ---
    xT = ode.invert(private_key, z0, is_forward=True)
    xT_key = lkt.hide(xT)

    if decoy_image_path is not None:   # C3 deniability (tuỳ chọn)
        decoy_img = np.array(Image.open(decoy_image_path).convert("RGB").resize((512, 512)))
        xT_decoy = ode.invert(public_key, ode.image2latent(decoy_img), is_forward=True)
        dm = DeniableMultiplexer(qkm.K_seed(), xT.numel(), subspace_ratio=0.5, beta=0.6)
        muxed = dm.mux(xT_key.detach().cpu().numpy().reshape(-1),
                       xT_decoy.detach().cpu().numpy().reshape(-1))
        xT_key = torch.from_numpy(muxed.reshape([int(s) for s in xT.shape])).to(device, xT.dtype)

    hide_latent = ode.invert(public_key, xT_key, is_forward=False)
    hide = ode.latent2image(hide_latent)
    hide_path = os.path.join(save_path, "hide.png")
    _save_rgb(hide_path, hide)

    # --- 5) SEAL (A2 OTP + C4 auth + C6 ecc) + Q5 quantum-money token ---
    pkg = os.path.join(qdir, "hide_quantum_package.npz")
    keyj = os.path.join(qdir, "hide_quantum_key.json")
    restored = os.path.join(qdir, "hide_from_quantum.png")
    sres = seal_encode(hide_path, pkg, keyj, qkm, check_rate=adapt["check_rate"],
                       ecc_repeat=adapt["ecc_repeat"], container_path_for_auth=hide_path)
    log(f"[SEAL] total_qubits={sres.total_qubits} ecc={sres.ecc_repeat} mac={sres.container_mac[:12]}..")

    token = mint_quantum_money(qkm, money_serial, hide_path, n_money=n_money)
    v_ok = verify_quantum_money(token, qkm, hide_path)
    forged = forge_quantum_money(token, seed=12345)
    v_forge = verify_quantum_money(forged, qkm, hide_path)
    log(f"[Q5-money] honest authentic={v_ok['authentic']} (qber={v_ok['qber']:.3f}) | "
        f"forged authentic={v_forge['authentic']} (qber={v_forge['qber']:.3f})")
    import json as _json
    Path(os.path.join(qdir, "quantum_money_token.json")).write_text(
        _json.dumps(token.to_dict(), indent=2), encoding="utf-8")

    if simulate_eve_package:
        st = simulate_eavesdropper_on_package(pkg, eve_interception_rate)
        log(f"[EVE-PKG] intercepted {st['intercepted']}/{st['total']} qubits")

    ures = seal_decode(pkg, keyj, restored, qkm, qber_threshold=0.11, gate_on_tamper=True,
                       container_path_for_auth=hide_path)
    log(f"[UNSEAL] qber={ures.qber:.4f} integrity_ok={ures.integrity_ok} auth_ok={ures.auth_ok}")

    # --- 6) Q6 SWAP fingerprint: hide gốc vs hide khôi phục từ gói lượng tử ---
    bits_hide = np.unpackbits(np.frombuffer(Path(hide_path).read_bytes(), np.uint8))
    bits_rest = np.unpackbits(np.frombuffer(Path(restored).read_bytes(), np.uint8))
    fp = fingerprint_compare(bits_hide, bits_rest)
    log(f"[Q6-fingerprint] hide vs restored: P(swap0)={fp['p_swap0']:.4f} | {fp['compression']}")

    # --- 7) REVEAL: B2 restore -> forward(public) -> unscramble -> backward(private) ---
    # B2: CHỈ khử nhiễu khi container THỰC SỰ qua kênh nhiễu (Eve/JPEG). Với ảnh SẠCH,
    # NLM denoise chỉ làm MỜ -> giảm fidelity reverse vô cớ. (Sửa nguyên nhân ② chất lượng.)
    _restored_rgb = _load_rgb(restored)
    hide_q = receiver_restore(_restored_rgb) if simulate_eve_package else _restored_rgb
    xT_key2 = ode.invert(public_key, ode.image2latent(hide_q), is_forward=True)
    xT2 = lkt.reveal(xT_key2)
    rev_latent = ode.invert(private_key, xT2, is_forward=False)
    reverse = ode.latent2image(rev_latent)
    rev_path = os.path.join(save_path, "reverse.png")
    _save_rgb(rev_path, reverse)

    result = {
        "gt": os.path.join(save_path, "gt.png"), "hide": hide_path,
        "restored_hide": restored, "reverse": rev_path, "quantum_money": token.to_dict()["serial"],
        "qkd": {"protocol": rep.protocol, "qber": rep.qber, "chsh_S": rep.chsh_S, "secure": rep.secure},
        "Q1_transform": lkt.kind, "Q4_quantum_key_mode": lkt.qmode,
        "Q2_helstrom": {"passive": hrep.is_passive, "p_detect_upper": hrep.p_detect_upper,
                        "advantage_upper": hrep.advantage_upper},
        "Q3_dfs": dfs,
        "Q5_money": {"honest_authentic": v_ok["authentic"], "honest_qber": v_ok["qber"],
                     "forged_authentic": v_forge["authentic"], "forged_qber": v_forge["qber"]},
        "Q6_fingerprint": fp,
        "seal": {"qber": ures.qber, "integrity_ok": ures.integrity_ok, "auth_ok": ures.auth_ok},
        "num_steps": num_steps, "squeeze_scale": squeeze_scale,
    }
    log("[DONE]")
    return result


# =====================================================================================
# ĐÁNH GIÁ: PSNR/SSIM + bảng tổng hợp Q-series
# =====================================================================================

def _psnr(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float64); b = b.astype(np.float64)
    if a.shape != b.shape:
        h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
        a, b = a[:h, :w], b[:h, :w]
    mse = np.mean((a - b) ** 2)
    return 100.0 if mse == 0 else float(20 * np.log10(255.0) - 10 * np.log10(mse))


def _ssim(a: np.ndarray, b: np.ndarray) -> float:
    try:
        from skimage.metrics import structural_similarity as ssim
        return float(ssim(a, b, channel_axis=2, data_range=255))
    except Exception:
        a = a.astype(np.float64); b = b.astype(np.float64)
        mu_a, mu_b, va, vb = a.mean(), b.mean(), a.var(), b.var()
        cov = ((a - mu_a) * (b - mu_b)).mean()
        c1, c2 = (0.01 * 255) ** 2, (0.03 * 255) ** 2
        return float(((2 * mu_a * mu_b + c1) * (2 * cov + c2)) /
                     ((mu_a ** 2 + mu_b ** 2 + c1) * (va + vb + c2)))


def evaluate_qseries(result: dict):
    """Bảng PSNR/SSIM: secret↔reverse (fidelity, muốn CAO) & secret↔hide (security, muốn THẤP)."""
    gt, hide, rev = _load_rgb(result["gt"]), _load_rgb(result["hide"]), _load_rgb(result["reverse"])
    rows = [
        {"cặp": "secret vs reverse [FIDELITY]", "PSNR": round(_psnr(gt, rev), 3),
         "SSIM": round(_ssim(gt, rev), 4), "kỳ vọng": "CAO (khôi phục tốt)"},
        {"cặp": "secret vs hide   [SECURITY]", "PSNR": round(_psnr(gt, hide), 3),
         "SSIM": round(_ssim(gt, hide), 4), "kỳ vọng": "THẤP (container khác secret)"},
    ]
    try:
        import pandas as pd
        df = pd.DataFrame(rows)
        try:
            from IPython.display import display; display(df)
        except Exception:
            print(df.to_string(index=False))
        return df
    except Exception:
        for r in rows:
            print(r)
        return rows


## Self-test Q1–Q6 (CPU) — chứng minh trước khi chạy SD


In [ ]:
# ============ SELF-TEST Q1–Q6 (chạy NGAY trên CPU, KHÔNG cần GPU) ============
# Chứng minh toàn bộ logic lượng tử/crypto đúng TRƯỚC khi chạy Stable Diffusion.
_selftest()


## Chạy pipeline Q-series (GPU)


In [ ]:
# ============ CHẠY PIPELINE Q-SERIES (cần GPU + Stable Diffusion v1.5) ============
QSERIES_CFG = dict(
    image_path="/kaggle/input/datasets/yuan1103/asserts/1.png",  # <-- ĐỔI sang ảnh secret của bạn
    private_key="Eiffel tower",     # khoá riêng (prompt forward)
    public_key="a tree",            # khoá công khai (prompt backward) -> nội dung container giả
    save_path="/kaggle/working/output_dpm",
    num_steps=25,              # DPM-Solver++ bậc 2: 25 bước ~ DDIM 50 bước
    passphrase="qseries-secret",    # passphrase -> sifted key (BB84/E91) -> mọi subkey
    protocol="bb84",                # 'bb84' | 'e91'
    inverter="dpm++",              # 'dpm++' (mới, khuyến nghị) | 'ddim' | 'edict'
    scramble_block=64,
    squeeze_scale=0.0,              # Q1/Q2: 0.0 = passive (vô hình, p_detect=0.5). Tăng (vd 0.3) để thấy trade-off.
    use_qddpm=True, qddpm_steps=4,  # Q4: chuỗi unitary sinh từ VQC
    eve_rate_key=0.0,               # >0: mô phỏng nghe lén KÊNH KHOÁ -> gate HUỶ giấu tin (C2/C4)
    simulate_eve_package=False,     # True: nghe lén GÓI lượng tử -> tamper-evidence (QBER tăng)
    eve_interception_rate=1.0,
    jpeg_probe_quality=75,          # Q3
    n_money=512,                    # Q5
    decoy_image_path=None,          # C3 deniability (tuỳ chọn): đường dẫn ảnh mồi
)

pipe = build_pipeline()
result = run_inference_qseries(pipe, **QSERIES_CFG)

import json as _json
print(_json.dumps(result, indent=2, ensure_ascii=False))


## Đánh giá & hiển thị


In [ ]:
# ============ ĐÁNH GIÁ + HIỂN THỊ KẾT QUẢ ============
import matplotlib.pyplot as plt

df = evaluate_qseries(result)   # PSNR/SSIM: fidelity (secret↔reverse) & security (secret↔hide)

fig, ax = plt.subplots(1, 4, figsize=(19, 5))
for a, key, title in zip(
    ax, ["gt", "hide", "restored_hide", "reverse"],
    ["secret (gt)", "hide (container GIẢ do SD sinh)", "restored hide", "reverse ≈ secret"],
):
    a.imshow(_load_rgb(result[key])); a.set_title(title); a.axis("off")
plt.tight_layout(); plt.show()

print("【Q2 Helstrom】", result["Q2_helstrom"])
print("【Q3 DFS    】", result["Q3_dfs"])
print("【Q5 money  】", result["Q5_money"])
print("【Q6 finger 】", result["Q6_fingerprint"])
print("【QKD       】", result["qkd"])
